# Queens 8 × 8 · Adaptive risk-budget sampling

Load the trained DFM checkpoint from `01_queens_discrete_flow_matching.ipynb`.
The sampler chooses its deletion count from a per-iteration risk budget and
stops when eight colored cells remain.

Inspect solution trajectories, measure success across repeated attempts and
record the number of model calls used by each trajectory.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT / "notebooks"))

import ipywidgets as widgets
import numpy as np
import plotly.graph_objects as go
import torch
from IPython.display import HTML, display
from plotly.subplots import make_subplots
from _shared.queens_data import BOARD_SIZE, EMPTY, load_queens_dataset, solution_checks
from _shared.queens_confidence_sampling import (
    load_trained_dfm,
    retry_statistics,
)
from _shared.queens_adaptive_sampling import sample_adaptive_dfm
from _shared.queens_sampling_plots import confidence_retry_plot

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARTIFACTS = PROJECT_ROOT / "artifacts" / "queens8"
CHECKPOINT_PATH = ARTIFACTS / "queens8_dfm.pt"
RESULT_PATH = ARTIFACTS / "queens8_dfm_adaptive_evaluation.npz"
PLOT_CONFIG = dict(responsive=True, displaylogo=False, displayModeBar=False)
NUM_CLASSES = BOARD_SIZE + 1
PALETTE = [
    "#B8A5E8",
    "#F0B48B",
    "#92C9E8",
    "#A9D4A4",
    "#E8CD83",
    "#ECA6BB",
    "#9FB9D4",
    "#C7B8A3",
    "#F1F3F5",
]
CATEGORICAL_SCALE = [
    [position, color]
    for index, color in enumerate(PALETTE)
    for position in (index / NUM_CLASSES, (index + 1) / NUM_CLASSES)
]

## 1. Load the trained model

Load the prepared dataset, trained weights and saved test split. The loader
checks the dataset fingerprint. Each model call receives the current grid and
scalar time.


In [2]:
puzzles = load_queens_dataset(PROJECT_ROOT / "data" / "queens8")
regions = puzzles["regions"]
model, splits, model_info = load_trained_dfm(CHECKPOINT_PATH, puzzles, device=DEVICE)
SEED = model_info["seed"]
TAUS = (0.25, 1.0, 4.0)
RISK_BUDGETS = (0.01, 0.025, 0.05, 0.1, 0.25, 0.5, 1.0)
display(
    HTML(
        f"<b>Loaded trained DFM</b> · {DEVICE} · {model_info['optimizer_updates']:,} updates<br>"
        f"{len(regions):,} puzzles · {len(splits['test']):,} held-out puzzles"
    )
)

In [3]:
def add_board(fig, tokens, board, row, col, queens=False):
    tokens, board = np.asarray(tokens), np.asarray(board)
    fig.add_trace(
        go.Heatmap(
            z=tokens,
            zmin=-0.5,
            zmax=NUM_CLASSES - 0.5,
            colorscale=CATEGORICAL_SCALE,
            showscale=False,
            xgap=1,
            ygap=1,
            customdata=board,
            hovertemplate="row=%{y}, col=%{x}<br>region=%{customdata}<br>token=%{z}<extra></extra>",
        ),
        row=row,
        col=col,
    )
    xs, ys = [], []
    for r in range(BOARD_SIZE):
        for c in range(BOARD_SIZE):
            if c + 1 < BOARD_SIZE and board[r, c] != board[r, c + 1]:
                xs.extend([c + 0.5, c + 0.5, None])
                ys.extend([r - 0.5, r + 0.5, None])
            if r + 1 < BOARD_SIZE and board[r, c] != board[r + 1, c]:
                xs.extend([c - 0.5, c + 0.5, None])
                ys.extend([r + 0.5, r + 0.5, None])
    fig.add_trace(
        go.Scatter(
            x=xs,
            y=ys,
            mode="lines",
            line=dict(color="#64748B", width=1.5),
            hoverinfo="skip",
            showlegend=False,
        ),
        row=row,
        col=col,
    )
    if queens:
        qr, qc = np.where(tokens != EMPTY)
        fig.add_trace(
            go.Scatter(
                x=qc,
                y=qr,
                mode="text",
                text=["♛"] * len(qr),
                textfont=dict(size=25, color="#172033"),
                hoverinfo="skip",
                showlegend=False,
            ),
            row=row,
            col=col,
        )
    fig.update_xaxes(
        range=[-0.5, BOARD_SIZE - 0.5],
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        constrain="domain",
        fixedrange=True,
        row=row,
        col=col,
    )
    xaxis_name = fig.get_subplot(row, col).xaxis.plotly_name.replace("axis", "")
    fig.update_yaxes(
        range=[BOARD_SIZE - 0.5, -0.5],
        showgrid=False,
        zeroline=False,
        showticklabels=False,
        scaleanchor=xaxis_name,
        scaleratio=1,
        constrain="domain",
        fixedrange=True,
        row=row,
        col=col,
    )


def style_boards(fig, title, rows=1):
    fig.update_layout(
        title=dict(text=title, x=0, font=dict(size=22)),
        template="plotly_white",
        autosize=True,
        height=rows * 225 + 90,
        margin=dict(l=10, r=10, t=85, b=20),
        font=dict(family="Arial, sans-serif", size=15),
        showlegend=False,
    )
    fig.update_annotations(font=dict(size=14))
    return fig

## 2. Adaptive deletion rule

Let $M$ be the current number of colored cells. Set time from the fraction of
required deletions already completed:

$$
t=\frac{64-M}{56}.
$$

The model predicts $p_i=P_\theta(x_1^i=\mathrm{EMPTY}\mid x_t,t)$ for each
colored cell. Generate a weighted random order $\pi$ without replacement using
weights $p_i^{1/\tau}$. Gumbel scores $\log(p_i)/\tau-\log(-\log u_i)$ implement
this ordering, where $u_i\sim U(0,1)$.

Erase the longest prefix whose cumulative estimated risk fits the budget:

$$
m=\max\left\{k\in\{0,\ldots,M-8\}:
\sum_{j=1}^{k}(1-p_{\pi_j})\leq\delta\right\}.
$$

The risk uses the model's original probabilities; temperature changes the order.
For calibrated probabilities, the sum bounds the probability of deleting at least
one queen in that prefix. The budget applies to one iteration. It is an estimate
of local risk, rather than a guarantee for the entire trajectory.

If $m=0$, erase the first cell in the sampled order and mark a **forced step**.
This step may exceed the risk budget. Each iteration deletes at least one cell
and at most $M-8$, so generation finishes with exactly eight cells in **1–56
model calls**. Each trajectory in a batch has its own deletion count and time;
finished trajectories are excluded from later model calls.

Evaluation checks one queen per row, column and region, and the non-touching rule.


## 3. Solution trajectories

Rows show temperatures for the same held-out puzzle at δ = 0.05. Panels are
sampled from the actual iteration history, including the initial and final states.
Labels show the model-call index, time and number of cells deleted. A star marks
a forced step. The final panel reports validity and total model calls.


In [4]:
N_TRAJECTORY_PUZZLES = 2
TRAJECTORY_RISK_BUDGET = 0.05
N_TRAJECTORY_PANELS = 6
for number, index in enumerate(splits["test"][:N_TRAJECTORY_PUZZLES]):
    board = regions[int(index)]
    runs = []
    for tau in TAUS:
        result = sample_adaptive_dfm(
            model,
            board,
            risk_budget=TRAJECTORY_RISK_BUDGET,
            tau=tau,
            seed=SEED + 600 + number,
            record_trajectory=True,
        )
        history = result["trajectory"]
        positions = np.rint(
            np.linspace(0, len(history) - 1, min(N_TRAJECTORY_PANELS, len(history)))
        ).astype(int)
        frames = [history[position] for position in positions]
        solved = bool(solution_checks(board, result["tokens"][0].numpy())["solved"])
        runs.append((tau, result, frames, solved))
    panel_count = max(len(frames) for _, _, frames, _ in runs)
    titles = []
    for tau, result, frames, solved in runs:
        for panel in range(panel_count):
            if panel >= len(frames):
                titles.append("")
                continue
            frame = frames[panel]
            if panel == 0:
                title = f"τ = {tau:g} · initial"
            elif panel == len(frames) - 1:
                title = (
                    "Solved" if solved else "Not solved"
                ) + f" · {int(result['iterations'][0])} calls"
            else:
                star = " *" if frame["forced"] else ""
                title = f"k = {frame['iteration']} · t = {frame['time']:.2f}<br>−{frame['deleted']} cells{star}"
            titles.append(title)
    fig = make_subplots(
        rows=len(TAUS),
        cols=panel_count,
        subplot_titles=titles,
        horizontal_spacing=0.025,
        vertical_spacing=0.12,
    )
    for row, (tau, result, frames, solved) in enumerate(runs, 1):
        for col, frame in enumerate(frames, 1):
            add_board(
                fig, frame["state"].numpy(), board, row, col, queens=col == len(frames)
            )
        for col in range(len(frames) + 1, panel_count + 1):
            fig.update_xaxes(visible=False, row=row, col=col)
            fig.update_yaxes(visible=False, row=row, col=col)
    style_boards(
        fig,
        f"Adaptive DFM · {puzzles['ids'][int(index)]} · δ = {TRAJECTORY_RISK_BUDGET:g}",
        rows=len(TAUS),
    ).show(config=PLOT_CONFIG)

## 4. Success probability and sampling cost

Evaluate three temperatures and seven risk budgets on the same 256 held-out
puzzles, with 32 stochastic attempts per combination and puzzle. Shared seeds
pair randomness across configurations. Each attempt starts from a full coloring.

For $c$ successes among $A$ attempts, average over subsets of $k$ attempts:

$$
\widehat{\mathrm{pass@}k}=1-\frac{\binom{A-c}{k}}{\binom{A}{k}},
\qquad 0\leq k\leq A.
$$

Average the estimates across puzzles. The 95% intervals bootstrap whole puzzles
using the same resampling weights for all configurations. Record model calls and
forced steps for every attempt alongside the validity outcome. The progress bar
counts completed cell deletions; each trajectory requires 56 deletions.


In [5]:
N_EVAL_PUZZLES = 256  # None evaluates every held-out puzzle.
ATTEMPTS_PER_PUZZLE = 32
EVAL_BATCH_SIZE = 128
EVAL_SEED = SEED + 10_000
if ATTEMPTS_PER_PUZZLE < 1 or EVAL_BATCH_SIZE < 1:
    raise ValueError("Attempt count and batch size must be positive.")
if N_EVAL_PUZZLES is not None and N_EVAL_PUZZLES < 1:
    raise ValueError("N_EVAL_PUZZLES must be positive or None.")
if (
    not TAUS
    or len(set(TAUS)) != len(TAUS)
    or any(not np.isfinite(t) or t <= 0 for t in TAUS)
):
    raise ValueError("Temperatures must be distinct finite positive values.")
if (
    not RISK_BUDGETS
    or len(set(RISK_BUDGETS)) != len(RISK_BUDGETS)
    or any(not np.isfinite(b) or b < 0 for b in RISK_BUDGETS)
):
    raise ValueError("Risk budgets must be distinct finite nonnegative values.")
evaluation_indices = np.random.default_rng(EVAL_SEED).permutation(splits["test"])
if N_EVAL_PUZZLES is not None:
    evaluation_indices = evaluation_indices[:N_EVAL_PUZZLES]
flat_indices = np.repeat(evaluation_indices, ATTEMPTS_PER_PUZZLE)
shape = (len(TAUS), len(RISK_BUDGETS), len(evaluation_indices), ATTEMPTS_PER_PUZZLE)
successes = np.empty(shape, dtype=bool)
model_calls = np.empty(shape, dtype=np.int64)
forced_steps = np.empty(shape, dtype=np.int64)
evaluation_complete = False
deletions_per_attempt = BOARD_SIZE**2 - BOARD_SIZE
combination_work = len(flat_indices) * deletions_per_attempt
evaluation_progress = widgets.IntProgress(
    value=0,
    max=combination_work * len(TAUS) * len(RISK_BUDGETS),
    description="Evaluation",
    layout=widgets.Layout(width="100%"),
)
evaluation_status = widgets.HTML()
display(evaluation_progress, evaluation_status)
completed_work = 0
for tau_index, tau in enumerate(TAUS):
    for budget_index, risk_budget in enumerate(RISK_BUDGETS):
        outcome_batches, call_batches, forced_batches = [], [], []
        for start in range(0, len(flat_indices), EVAL_BATCH_SIZE):
            indices = flat_indices[start : start + EVAL_BATCH_SIZE]
            evaluation_status.value = (
                f"τ = {tau:g} · δ = {risk_budget:g} · "
                f"combination {tau_index * len(RISK_BUDGETS) + budget_index + 1}/"
                f"{len(TAUS) * len(RISK_BUDGETS)} · {start:,}/{len(flat_indices):,} trajectories"
            )

            def update_evaluation(done, total, start=start, offset=completed_work):
                evaluation_progress.value = (
                    offset + start * deletions_per_attempt + done
                )

            result = sample_adaptive_dfm(
                model,
                regions[indices],
                risk_budget=risk_budget,
                tau=tau,
                seed=EVAL_SEED + start,
                progress=update_evaluation,
            )
            outcome_batches.append(
                solution_checks(regions[indices], result["tokens"].numpy())["solved"]
            )
            call_batches.append(result["iterations"].numpy())
            forced_batches.append(result["forced_steps"].numpy())
        target_shape = (len(evaluation_indices), ATTEMPTS_PER_PUZZLE)
        successes[tau_index, budget_index] = np.concatenate(outcome_batches).reshape(
            target_shape
        )
        model_calls[tau_index, budget_index] = np.concatenate(call_batches).reshape(
            target_shape
        )
        forced_steps[tau_index, budget_index] = np.concatenate(forced_batches).reshape(
            target_shape
        )
        completed_work += combination_work
        evaluation_progress.value = completed_work
evaluation_complete = True
evaluation_progress.bar_style = "success"
evaluation_status.value = (
    f"Completed {len(TAUS) * len(RISK_BUDGETS)} combinations · "
    f"{len(evaluation_indices):,} puzzles · {ATTEMPTS_PER_PUZZLE} attempts each · "
    f"{model_calls.sum():,} total model calls"
)
evaluation_metadata = dict(
    taus=np.asarray(TAUS),
    risk_budgets=np.asarray(RISK_BUDGETS),
    dataset_indices=evaluation_indices,
    puzzle_ids=puzzles["ids"][evaluation_indices],
    evaluation_seed=EVAL_SEED,
    batch_size=EVAL_BATCH_SIZE,
    checkpoint_sha256=model_info["checkpoint_sha256"],
    dataset_sha256=puzzles["metadata"]["data_sha256"],
    optimizer_updates=model_info["optimizer_updates"],
    sampler_version="queens8_adaptive_risk_v1",
    total_erasures=deletions_per_attempt,
)
np.savez_compressed(
    RESULT_PATH,
    successes=successes,
    model_calls=model_calls,
    forced_steps=forced_steps,
    **evaluation_metadata,
)

IntProgress(value=0, description='Evaluation', layout=Layout(width='100%'), max=9633792)

HTML(value='')

### Success across attempts

Color identifies temperature and line style identifies risk budget. Use the
independent selector groups to display their combinations. Layers controls the
95% intervals. Selection updates visibility while retaining the recorded results.


In [6]:
if not evaluation_complete:
    raise RuntimeError("Finish the evaluation cell before plotting.")
statistics = retry_statistics(successes, bootstrap_samples=3000, seed=EVAL_SEED + 1)
np.savez_compressed(
    RESULT_PATH,
    successes=successes,
    model_calls=model_calls,
    forced_steps=forced_steps,
    **evaluation_metadata,
    attempt_counts=statistics["attempts"],
    pass_at_k=statistics["mean"],
    lower=statistics["lower"],
    upper=statistics["upper"],
)
adaptive_view = confidence_retry_plot(
    statistics,
    TAUS,
    RISK_BUDGETS,
    budget_label="Risk budget δ",
    budget_name="δ",
    previous=globals().get("adaptive_view"),
)
display(adaptive_view)

SamplingComparison(children=(FigureWidget({
    'data': [{'fillcolor': 'rgba(117,167,255,0.07)',
             …

### Model calls per attempt

Mean model calls measure sampling cost. Error bars show the 5th–95th percentiles
across trajectories. Hover also reports the fraction of calls that forced a
single deletion because the sampled prefix exceeded the risk budget.


In [7]:
cost_fig = go.Figure()
cost_colors = ("#75A7FF", "#F4B66A", "#6DD3AC")
for tau_index, tau in enumerate(TAUS):
    means = model_calls[tau_index].mean(axis=(-2, -1))
    low, high = np.quantile(model_calls[tau_index], [0.05, 0.95], axis=(-2, -1))
    forced_fraction = forced_steps[tau_index].sum(axis=(-2, -1)) / model_calls[
        tau_index
    ].sum(axis=(-2, -1))
    cost_fig.add_scatter(
        x=RISK_BUDGETS,
        y=means,
        name=f"τ = {tau:g}",
        mode="lines+markers",
        line=dict(color=cost_colors[tau_index % len(cost_colors)], width=2.5),
        error_y=dict(
            type="data",
            symmetric=False,
            array=np.maximum(high - means, 0),
            arrayminus=np.maximum(means - low, 0),
        ),
        customdata=np.column_stack((low, high, 100 * forced_fraction)),
        hovertemplate=(
            "δ = %{x:g}<br>Mean calls: %{y:.2f}<br>5–95%: [%{customdata[0]:.0f}, %{customdata[1]:.0f}]"
            "<br>Forced calls: %{customdata[2]:.1f}%<extra>%{fullData.name}</extra>"
        ),
    )
cost_fig.update_layout(
    template="plotly_dark",
    paper_bgcolor="#1f1f1f",
    plot_bgcolor="#1f1f1f",
    title="Model calls per attempt",
    height=460,
    autosize=True,
    xaxis=dict(
        title="Risk budget δ", type="log" if min(RISK_BUDGETS) > 0 else "linear"
    ),
    yaxis=dict(title="Model calls", range=[0, deletions_per_attempt + 1]),
    font=dict(family="Arial, sans-serif", size=16),
    legend=dict(orientation="h", y=-0.2),
    margin=dict(l=75, r=25, t=65, b=100),
)
cost_fig.show(config=PLOT_CONFIG)